# ChurnSignal — Milestone 2: Exploratory Data Analysis (EDA)
**Learning-First SaaS Customer Retention Intelligence**

Welcome to **Milestone 2**! In this stage, we move beyond raw reconnaissance into **Exploratory Data Analysis (EDA)**.

---

### Why EDA is the Foundation of Applied ML:
> *"Algorithms do not understand business context—they only optimize loss functions over the distributions you give them."*

By exploring univariate spreads, bivariate contrasts, and correlation structures, we discover:
1. Which features carry real **predictive signal**.
2. Where **extreme skewness** or **outliers** exist that could disrupt models like Logistic Regression.
3. How customer behaviors shift prior to a churn event.
4. How business segments (like subscription plans) differ in stability.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display

# Add project root to Python path
project_root = Path('..').resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data.loader import load_raw_dataset, load_prediction_eligible_data

# Formatting
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', lambda x: '%.4f' % x)
print('Libraries loaded and project path configured.')

## 1. Load Dataset
We load the monthly panel dataset using our modular `src.data.loader` package.


In [ ]:
df = load_raw_dataset()
print(f'Total Customer-Month Records: {df.shape[0]:,}')
print(f'Total Features: {df.shape[1]}')
df.head(3)

## 2. Target Variable & Class Imbalance Analysis

### Concept Intuition:
In SaaS churn, churn is a **rare event**. Out of 79,817 customer months, only 1,056 records represent an account about to churn next month (~1.32%).

**Why this matters**:
- An accuracy of **98.68%** is achieved simply by predicting that *nobody ever churns*.
- Therefore, we must never optimize for standard accuracy.
- We will look at **Precision-Recall curves (PR-AUC)** and **Precision@Top-K** in Milestone 5.


In [ ]:
display(Image(filename='../reports/figures/01_target_imbalance.png'))

churn_counts = df['churned_next_month'].value_counts()
churn_pct = df['churned_next_month'].value_counts(normalize=True) * 100
pd.DataFrame({'Count': churn_counts, 'Percentage (%)': churn_pct})

## 3. Univariate Feature Distributions

### Concept Intuition:
Examining features individually reveals their central tendency (mean vs median), spread (standard deviation), and tail skewness.

- **Skewed counts**: `sessions` and `active_seats` have long right-hand tails (a few enterprise teams with high activity).
- **Zero-seat accounts**: A notable cluster of records has `active_seats == 0` (accounts paying but not logging in).
- **Bounded indices**: `feature_usage_score` (0-100) and `nps_score` (0-10) are bounded scales.


In [ ]:
display(Image(filename='../reports/figures/02_univariate_distributions.png'))

key_cols = ['sessions', 'feature_usage_score', 'active_seats', 'mrr', 'nps_score', 'tenure_month']
df[key_cols].describe().T[['mean', 'std', 'min', '50%', 'max']]

## 4. Bivariate Analysis: Retained vs. Churning Customers

### Concept Intuition:
To predict churn, we must understand how customer behavior diverges before churn happens.
Comparing active accounts that stay (`churned_next_month == 0`) with those that churn (`churned_next_month == 1`) reveals powerful leading indicators:

1. **Feature Usage Collapse**: ~42% drop in average usage score.
2. **Session Drop**: ~36% drop in logins/sessions.
3. **NPS Plunge**: Average satisfaction score falls from 4.7 to 2.7.
4. **Billing Friction**: Payment failures triple prior to churn.
5. **Support Tickets**: Modest increase in support friction.


In [ ]:
display(Image(filename='../reports/figures/03_bivariate_comparisons.png'))

bivariate_summary = df.groupby('churned_next_month')[
    ['feature_usage_score', 'sessions', 'nps_score', 'support_tickets', 'payment_failures', 'active_seats']
].mean().T

bivariate_summary.columns = ['Retained Mean (0)', 'Churn Mean (1)']
bivariate_summary['Absolute Change'] = bivariate_summary['Churn Mean (1)'] - bivariate_summary['Retained Mean (0)']
bivariate_summary['Percentage Change (%)'] = (bivariate_summary['Absolute Change'] / bivariate_summary['Retained Mean (0)']) * 100
bivariate_summary

## 5. Subscription Plan Breakdown (`plan_type`)

### Concept Intuition:
Different pricing tiers serve different customer segments. Are Enterprise accounts more stable than Starter accounts?

- **Starter Plan**: Highest churn rate (~1.6%). Smaller teams with lower commitment.
- **Enterprise Plan**: Lowest churn rate (~0.8%). Dedicated account management and annual commitments create higher retention stability.


In [ ]:
display(Image(filename='../reports/figures/04_plan_type_analysis.png'))

plan_summary = df.groupby('plan_type', observed=False).agg(
    total_accounts=('user_id', 'count'),
    churned_accounts=('churned_next_month', 'sum'),
    churn_rate_pct=('churned_next_month', lambda x: x.mean() * 100)
).sort_values(by='churn_rate_pct', ascending=False)
plan_summary

## 6. Correlation Analysis & Collinearity Detection

### Concept Intuition:
Correlation measures linear association between features ($r \in [-1, 1]$).

- **Exact Collinearity Trap**: `monthly_price` and `mrr` have $r = 1.000$ (identical values everywhere). In linear models, this causes severe numerical instability.
- **Natural Team Clusters**: `active_seats`, `mrr`, and `sessions` are strongly co-linear ($r > 0.70$) because larger teams naturally pay more and generate more logins.
- **Early Warning Signals**: `feature_usage_score` ($r = -0.11$) and `nps_score` ($r = -0.10$) have the strongest negative correlation with churn.


In [ ]:
display(Image(filename='../reports/figures/05_correlation_heatmap.png'))

corr_with_churn = df.select_dtypes(include=[np.number]).corr()['churned_next_month'].sort_values()
pd.DataFrame({'Correlation with churned_next_month': corr_with_churn})

## 7. Macro Temporal Trajectory (24-Month Timeline)

### Concept Intuition:
A SaaS customer base undergoes distinct lifecycle phases:
- **Phase 1: Onboarding Growth (Jan 2024 to Jan 2025)**: Active users scale from 390 to 4,615 as new customer cohorts join each month.
- **Phase 2: Pure Retention / Aging (Feb 2025 to Dec 2025)**: No new users join; active users gently decline to 3,931 as churn occurs.
- **Right-Censoring Trap**: In December 2025, churn drops to 0.00% because January 2026 was never recorded. This confirms why December 2025 must be excluded from evaluation.


In [ ]:
display(Image(filename='../reports/figures/06_temporal_dynamics.png'))

## 8. Milestone 2 Key Insights & Next Steps

### Key Discoveries from EDA:
1. **Behavioral Divergence**: Customers on the brink of churn show clear behavioral decline: feature usage drops by 42%, logins drop by 36%, and satisfaction drops by 43%.
2. **Billing Alarm**: Payment failures triple before churn, indicating technical/credit friction as a major leading indicator.
3. **Plan Disparity**: Starter accounts churn at nearly double the rate of Enterprise accounts.
4. **Collinearity Resolved**: `monthly_price` must be dropped in favor of `mrr`.
5. **Static Features Miss Velocity**: Static monthly metrics only show current status. A customer dropping from 50 sessions to 25 is at much higher risk than a customer who always logged 25 sessions. 

### Transition to Milestone 3: Feature Engineering
In the next milestone, we will build **rolling momentum and velocity features**:
$$\Delta \text{sessions} = \text{sessions}_T - \text{sessions}_{T-1}$$
$$\text{usage\_ratio} = \frac{\text{feature\_usage}_T}{\text{mean}(\text{feature\_usage}_{T-2..T})}$$
